# Formally Verified Delta-Hedging: A Demonstration

This notebook demonstrates the Options Hedge Engine: a discrete delta-hedging
simulation where **every portfolio state transition carries a machine-checked proof
of correctness**.

The engine routes every trade through a Lean 4 accounting kernel whose theorems
are proven correct for *all possible inputs* — not just the test cases. At each
rebalancing step, a `StepCertificate` is emitted verifying that `valueUpdateFormula`
held. A bug raises `ValueError` immediately rather than silently corrupting the result.

## Contents

1. [Hull Table 19.2 Replication](#hull-192) — week-by-week delta-hedging of a written call
2. [Step Certificate Audit Trail](#certificates) — all 20 certificates displayed
3. [What a Bug Looks Like](#bug-demo) — deliberately broken accounting, certificate catches it
4. [Human–AI Collaboration via Lean](#human-ai) — the development workflow
5. [Portfolio Delta-Hedging: Short Straddle](#straddle) — multi-leg portfolio, net delta tracking
6. [OptionMetrics Real-Data Pipeline](#optionmetrics) — ETL from WRDS to `PricePath`, SPY ATM options

In [ ]:
# Standard setup
import sys
import os

# Ensure the Python package is importable from the notebooks/ directory
sys.path.insert(
    0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "python", "src")
)

from hedge_engine.backtest.runner import run_delta_hedge, run_portfolio_hedge, OptionLeg
from hedge_engine.backtest.scenarios import (
    hull_192_path,
    HULL_192_K,
    HULL_192_R,
    HULL_192_SIGMA,
    HULL_192_N_CONTRACTS,
    STRADDLE_K,
    STRADDLE_R,
    STRADDLE_SIGMA,
    STRADDLE_N_CONTRACTS,
)
from hedge_engine.backtest.audit import verify_step
from hedge_engine.pricer.black_scholes import bs_price, bs_greeks

print("Imports OK")

<a id="hull-192"></a>
## 1. Hull Table 19.2 Replication

From Hull *Options, Futures, and Other Derivatives* (12th ed.), Table 19.2:

| Parameter | Value |
|-----------|-------|
| Underlying | S₀ = \$49 |
| Strike | K = \$50 |
| Rate | r = 5% p.a. |
| Volatility | σ = 20% p.a. |
| Maturity | T = 20 weeks |
| Position | Short 100,000 calls |

The writer receives the option premium upfront and delta-hedges weekly.
The total cost of running the hedge (rebalancing costs minus premium retained)
is known from Hull to be approximately **\$263,300**.

Every step goes through the Lean kernel; every step is certified.

In [ ]:
# ── Run the backtest ──────────────────────────────────────────────────────
path = hull_192_path()
result = run_delta_hedge(
    path=path,
    K=HULL_192_K,
    r=HULL_192_R,
    sigma=HULL_192_SIGMA,
    n_contracts=HULL_192_N_CONTRACTS,
)

print(f"Total hedging cost : ${result.total_hedging_cost:>12,.0f}")
print("Hull Table 19.2    : $       263,300  (reference)")
print(
    f"Difference         : {(result.total_hedging_cost - 263_300) / 263_300 * 100:+.1f}%"
)
print(f"Certificates issued: {len(result.certificates)}")
print(f"Certificates passed: {sum(c.invariant_holds for c in result.certificates)}")

In [ ]:
# ── Build the week-by-week P&L table ─────────────────────────────────────
K = HULL_192_K
r = HULL_192_R
sigma = HULL_192_SIGMA
n_contracts = HULL_192_N_CONTRACTS
T_total = 20 / 52

rows = []
prev_hedge = 0
for week, (t, S) in enumerate(zip(path.times, path.prices)):
    T_rem = T_total - t
    if T_rem <= 0:
        T_rem = 1e-10  # avoid division by zero at expiry

    delta = bs_greeks(S=S, K=K, T=T_rem, r=r, sigma=sigma, option_type="call").delta
    hedge_qty = round(delta * n_contracts)
    opt_price = bs_price(S=S, K=K, T=T_rem, r=r, sigma=sigma, option_type="call").value

    shares_purchased = hedge_qty - prev_hedge
    rows.append(
        {
            "Week": week,
            "S ($)": S,
            "Delta": round(delta, 4),
            "Hedge (shares)": hedge_qty,
            "Bought/(sold)": shares_purchased,
            "Cost of shares ($)": round(shares_purchased * S, 0),
            "Option price ($)": round(opt_price, 4),
        }
    )
    prev_hedge = hedge_qty

# Print as a formatted table
header = f"{'Wk':>3}  {'S ($)':>8}  {'Delta':>7}  {'Hedge':>9}  {'Bought':>9}  {'Cost ($)':>12}  {'Opt ($)':>9}"
print(header)
print("-" * len(header))
for row in rows:
    print(
        f"{row['Week']:>3}  "
        f"{row['S ($)']:>8.2f}  "
        f"{row['Delta']:>7.4f}  "
        f"{row['Hedge (shares)']:>9,}  "
        f"{row['Bought/(sold)']:>+9,}  "
        f"{row['Cost of shares ($)']:>12,.0f}  "
        f"{row['Option price ($)']:>9.4f}"
    )

<a id="certificates"></a>
## 2. Step Certificate Audit Trail

At each rebalancing step the engine emits a `StepCertificate` that verifies
Lean's `valueUpdateFormula`:

$$\Delta\text{PV} = \text{pre-trade qty} \times (\text{exec price} - \text{mark before}) - \text{fee}$$

All values are in basis points (×10,000). The certificate checks that the
*observed* ΔPV equals the *predicted* ΔPV. A mismatch would mean the accounting
kernel computed a different number than the theorem guarantees — which is
impossible as long as the kernel is the proven Lean code.

In [ ]:
# ── Display the audit trail ───────────────────────────────────────────────
certs = result.certificates

header = f"{'Step':>4}  {'PV before (bp)':>15}  {'PV after (bp)':>14}  {'ΔPV':>10}  {'Expected':>10}  {'OK?':>5}"
print(header)
print("-" * len(header))
for c in certs:
    ok = "✓" if c.invariant_holds else "✗ FAIL"
    print(
        f"{c.step:>4}  "
        f"{c.portfolio_value_before:>15,}  "
        f"{c.portfolio_value_after:>14,}  "
        f"{c.delta_pv:>+10,}  "
        f"{c.expected_delta_pv:>+10,}  "
        f"{ok:>5}"
    )

print()
n_pass = sum(c.invariant_holds for c in certs)
print(f"Result: {n_pass}/{len(certs)} certificates passed")

<a id="bug-demo"></a>
## 3. What a Bug Looks Like

The question a practitioner always asks: **"what bug does this actually prevent?"**

Here is a concrete example. Suppose there is an off-by-one error in the accounting
layer — for instance, a trade that should move the portfolio value by +2,500 bp
is instead recorded as moving it by +1,000 bp (the wrong value).

The `StepCertificate` catches this immediately: `invariant_holds = False`, and
the runner raises `ValueError` with a diagnostic. The bug cannot accumulate silently
across 20 weeks and produce a plausible but wrong final hedging cost.

In [ ]:
# ── Demonstrate a bug being caught ────────────────────────────────────────
#
# Scenario: we hold 1 share of an asset marked at $100.00.
# We execute a trade at $100.25 (25 bp above mark), fee=0.
# valueUpdateFormula predicts: ΔPV = 1 × (100.25 - 100.00) × 10000 = +2500 bp
#
# A buggy accounting layer computes the wrong PV after the trade:
# it reports PV went from 1,000,000 bp to 1,001,000 bp (+1000 bp instead of +2500 bp).

from hedge_engine.pricer.conventions import to_bp

print("=" * 60)
print("SIMULATING A BUGGY ACCOUNTING STEP")
print("=" * 60)
print()
print("Setup:")
print("  pre_trade_qty  = 1 share")
print("  exec_price     = $100.25  →", to_bp(100.25), "bp")
print("  mark_before    = $100.00  →", to_bp(100.00), "bp")
print("  fee            = $0")
print()
print("Lean's valueUpdateFormula predicts:")
expected = 1 * (to_bp(100.25) - to_bp(100.00)) - 0
print(f"  ΔPV = 1 × ({to_bp(100.25)} − {to_bp(100.00)}) − 0 = +{expected} bp")
print()

# The bug: the accounting layer reports +1000 bp instead of +2500 bp
pv_before = 1_000_000
pv_after_buggy = pv_before + 1_000  # BUG: should be +2500

print(
    f"Buggy accounting reports: PV {pv_before:,} → {pv_after_buggy:,} (ΔPV = +1,000 bp)"
)
print()

try:
    cert = verify_step(
        pv_before=pv_before,
        pv_after=pv_after_buggy,
        pre_trade_qty=1,
        exec_price_bp=to_bp(100.25),
        mark_before_bp=to_bp(100.00),
        fee_bp=0,
        step=0,
    )
    print("ERROR: no exception raised — this should not happen")
except ValueError as e:
    print("BUG CAUGHT — ValueError raised immediately:")
    print(f"  {e}")
    print()
    print("The engine halted with a diagnostic.")
    print("No silent wrong result. No plausible-but-incorrect hedging cost.")

In [ ]:
# ── For completeness: show the correct step passing ───────────────────────
print("For comparison — the same step with correct accounting:")
print()

pv_after_correct = pv_before + expected  # +2500 bp (correct)

cert = verify_step(
    pv_before=pv_before,
    pv_after=pv_after_correct,
    pre_trade_qty=1,
    exec_price_bp=to_bp(100.25),
    mark_before_bp=to_bp(100.00),
    fee_bp=0,
    step=0,
)

print(f"  PV: {pv_before:,} → {pv_after_correct:,} (ΔPV = +{expected:,} bp)")
print(f"  invariant_holds = {cert.invariant_holds}  ✓")
print()
print("This is what all 20 steps in Hull Table 19.2 look like.")

<a id="human-ai"></a>
## 4. Human–AI Collaboration via Lean

This project is built with an AI coding assistant (Claude). The Lean proof system
plays a second, less obvious role beyond runtime verification: it acts as a **formal
development scaffold** that constrains what the AI can generate.

### The development loop

```
1. Human specifies WHAT MUST BE TRUE   →  the theorem
2. AI generates code THAT MUST SATISFY IT  →  the implementation
3. Lean verifies the combination is correct  →  proof compiles or it doesn't
4. Human reviews THEOREM STATEMENTS, not implementation line-by-line
```

### Why this matters

A unit test checks one input. A Lean proof checks all inputs. The theorems in
`Invariants.lean` and `OptionInvariants.lean` hold for **every possible portfolio,
every possible trade, every possible option strike and spot price** — not just
the scenarios we thought to test.

The AI cannot introduce a silent accounting error that passes the formal spec.
It would need to change the theorem to do so — a visible, reviewable act.
The human's oversight concentrates at the level of mathematical claims
rather than implementation details.

### The theorems proven (kernel)

| Theorem | Economic meaning |
|---------|------------------|
| `valueIdentity` | Portfolio value = cash + Σ(qty × mark), always |
| `valueUpdateFormula` | ΔPV = pre-trade qty × (exec − mark) − fee |
| `selfFinancing` | Trading at the mark changes PV only by the fee |
| `quantityConservation` | Shares cannot appear from thin air |
| `cashUpdateCorrect` | Every dollar spent flows through cash (proved `rfl`) |
| `putCallParity` | Call payoff − put payoff = spot − strike, exactly |
| `settlement_value_formula` | ΔPV = qty × (payoff − mark) at expiry, ITM and OTM unified |

### Zero-sorry discipline

Lean's `sorry` tactic allows a proof to be skipped. This project maintains
**zero sorry** across all files. The human's responsibility is to verify this
before any merge — `lake build` reports any occurrence. This is the simplest
possible audit: one command, binary output.

### Step certificates as the audit trail

The `StepCertificate` objects shown above are not just a test artifact.
They are a machine-checkable record that `valueUpdateFormula` held at every
step of every backtest run. Any third party with a Lean installation can
verify the proof from source. The audit trail is the proof obligation plus
the runtime certificates — both are public and reproducible.

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────
print("Engine summary")
print("=" * 40)
print(f"  Hull 19.2 hedging cost : ${result.total_hedging_cost:>10,.0f}")
print("  Hull reference         : $   263,300")
print(f"  Step certificates      : {len(result.certificates):>10}")
print(
    f"  All passed             : {all(c.invariant_holds for c in result.certificates)!s:>10}"
)
print(f"  Sorry count            : {'0 (zero-sorry discipline)':>10}")
print()
print("Theorems checked at every step: valueUpdateFormula")
print("Theorems checked at expiry    : settlement_value_formula")

<a id="straddle"></a>
## 5. Portfolio Delta-Hedging: Short Straddle

A **short straddle** (short call + short put at the same strike) is one of the most
common positions in options market-making. The straddle writer collects both premiums
upfront and runs a delta hedge to neutralise directional risk.

Key characteristics of the short straddle:
- **Net delta ≈ 0** initially (when S₀ ≈ K), so the initial hedge is small
- **Delta grows** as the underlying moves away from strike — the hedge rebalances more
- **Net gamma is negative** (short gamma): large moves are costly; the hedger
  pays more in rebalancing than expected from BS
- **Net theta is positive**: time decay earns premium for the writer

We use the same Hull 19.2 price path (S₀=49 → S_T=57.25) with a written straddle
at K=50. The call expires ITM; the put expires OTM.

| Leg | Strike | n_contracts | Net delta at t=0 |
|-----|--------|-------------|-------------------|
| Short call | \$50 | −100,000 | call_Δ × (−100k) < 0 |
| Short put  | \$50 | −100,000 | put_Δ × (−100k) > 0 |
| **Net** | | | **≈ −4,000 shares** (S₀ < K) |

The underlying hedge is long ~4,000 shares initially (small offset), growing
as the call goes deeper ITM.

In [ ]:
# ── Run the straddle backtest ─────────────────────────────────────────────
straddle_legs = [
    OptionLeg(
        option_id="CALL_K50",
        option_type="call",
        K=STRADDLE_K,
        sigma=STRADDLE_SIGMA,
        n_contracts=-STRADDLE_N_CONTRACTS,  # short
    ),
    OptionLeg(
        option_id="PUT_K50",
        option_type="put",
        K=STRADDLE_K,
        sigma=STRADDLE_SIGMA,
        n_contracts=-STRADDLE_N_CONTRACTS,  # short
    ),
]

straddle_result = run_portfolio_hedge(
    path=hull_192_path(),
    legs=straddle_legs,
    r=STRADDLE_R,
)

n_pass = sum(c.invariant_holds for c in straddle_result.certificates)
print(f"Straddle hedging cost  : ${straddle_result.total_hedging_cost:>12,.0f}")
print(f"Step certificates      : {len(straddle_result.certificates):>12}")
print(
    f"Certificates passed    : {n_pass:>12} (all = {n_pass == len(straddle_result.certificates)})"
)

In [ ]:
# ── Week-by-week straddle net delta and hedge ─────────────────────────────
path_s = hull_192_path()
T_total_s = path_s.times[-1]

print(
    f"{'Wk':>3}  {'S ($)':>8}  {'Call Δ':>8}  {'Put Δ':>8}  {'Net opt Δ':>10}  {'Hedge (shs)':>12}"
)
print("-" * 65)
for week, (t, S) in enumerate(zip(path_s.times, path_s.prices)):
    T_rem = max(T_total_s - t, 1e-10)
    call_d = bs_greeks(
        S=S,
        K=STRADDLE_K,
        T=T_rem,
        r=STRADDLE_R,
        sigma=STRADDLE_SIGMA,
        option_type="call",
    ).delta
    put_d = bs_greeks(
        S=S,
        K=STRADDLE_K,
        T=T_rem,
        r=STRADDLE_R,
        sigma=STRADDLE_SIGMA,
        option_type="put",
    ).delta
    # Option portfolio delta (signed by contract count)
    opt_delta = (call_d + put_d) * (-STRADDLE_N_CONTRACTS)  # short both legs
    hedge_qty = round(-opt_delta)  # hedge negates option portfolio delta
    print(
        f"{week:>3}  {S:>8.2f}  {call_d:>8.4f}  {put_d:>8.4f}  {opt_delta:>+10,.0f}  {hedge_qty:>+12,}"
    )

<a id="optionmetrics"></a>
## 6. OptionMetrics Real-Data Pipeline

For a production-quality backtest, replace GBM-simulated or Hull paths with
real market data from **OptionMetrics via WRDS** (academic subscription).

### Data schema (OptionMetrics `opprcd` table)

The parquet file `data/spy_atm_options.parquet` should contain:

| Column | Source | Description |
|--------|--------|-------------|
| `underlying_ticker` | constant `"SPY"` | Underlying ticker |
| `date` | `opprcd.date` | Observation date (YYYY-MM-DD) |
| `expiry` | `opprcd.exdate` | Option expiry (YYYY-MM-DD) |
| `strike` | `opprcd.strike_price` | Strike price (dollars) |
| `option_type` | `opprcd.cp_flag` | `"C"` or `"P"` |
| `best_bid` | `opprcd.best_bid` | Best bid (dollars) |
| `best_offer` | `opprcd.best_offer` | Best offer (dollars) |
| `impl_volatility` | `opprcd.impl_volatility` | IV (fraction, e.g. 0.20) |
| `underlying_price` | `opprcd.spotprice` | Underlying spot price (dollars) |

### WRDS query (run in WRDS SAS or Python)

```sql
-- Step 1: Find SPY secid
SELECT secid FROM optionm.secnmd WHERE ticker = 'SPY' LIMIT 1;

-- Step 2: Download ATM SPY calls, ~1-month expiry, 2023-2024
SELECT date, exdate, strike_price, cp_flag,
       best_bid, best_offer, impl_volatility, spotprice
FROM optionm.opprcd
WHERE secid = <SPY_SECID>
  AND cp_flag = 'C'
  AND date BETWEEN '2023-01-01' AND '2024-12-31'
  AND exdate - date BETWEEN 20 AND 40
  AND ABS(strike_price / spotprice - 1) <= 0.03
  AND best_bid > 0 AND best_offer IS NOT NULL
  AND impl_volatility IS NOT NULL
ORDER BY date, exdate, strike_price;
```

Rename `exdate→expiry`, `strike_price→strike`, `cp_flag→option_type`,
`spotprice→underlying_price`, and add `underlying_ticker='SPY'`.
Save as `data/spy_atm_options.parquet`.

The cell below demonstrates the full ETL pipeline on **synthetic data**
that mirrors the OptionMetrics schema. Replace the synthetic `df` with
`pd.read_parquet('../data/spy_atm_options.parquet')` when real data arrives.

In [ ]:
# ── ETL pipeline demo (synthetic WRDS-like data) ─────────────────────────
# Replace synthetic_df with pd.read_parquet('../data/spy_atm_options.parquet')
# once you have downloaded the WRDS OptionMetrics data.

import pandas as pd
from hedge_engine.etl.wrds_loader import optionmetrics_option_snapshots_from_df
from hedge_engine.backtest.data_types import PricePath

# Synthetic SPY ATM call: 21 daily observations, spot ~$490, strike $490, 1-month expiry
_dates = [f"2024-01-{d:02d}" for d in range(2, 23)]
_spots = [490.0 + i * 0.8 - 0.3 * (i % 3) for i in range(21)]  # gentle uptrend
synthetic_df = pd.DataFrame(
    {
        "underlying_ticker": ["SPY"] * 21,
        "date": _dates,
        "expiry": ["2024-02-23"] * 21,
        "strike": [490.0] * 21,
        "option_type": ["C"] * 21,
        "best_bid": [max(0.05, 8.5 - i * 0.2) for i in range(21)],
        "best_offer": [max(0.10, 8.7 - i * 0.2) for i in range(21)],
        "impl_volatility": [0.155 + 0.001 * i for i in range(21)],
        "underlying_price": _spots,
    }
)

# Parse via the ETL loader (same path as real WRDS data)
snaps = optionmetrics_option_snapshots_from_df(synthetic_df)
print(f"Loaded {len(snaps)} option snapshots")
print(f"Date range  : {snaps[0].date} → {snaps[-1].date}")
print(f"Strike      : ${snaps[0].strike:.0f}")
print(
    f"Spot range  : ${snaps[0].underlying_price:.2f} → ${snaps[-1].underlying_price:.2f}"
)
print(f"IV range    : {snaps[0].implied_vol:.3f} → {snaps[-1].implied_vol:.3f}")
print(f"Mid-price   : ${snaps[0].mid_price:.2f} → ${snaps[-1].mid_price:.2f}")

In [ ]:
# ── Run the backtest on the ETL output ────────────────────────────────────
import pandas as pd

first = snaps[0]
T_total_etl = (pd.Timestamp(first.expiry) - pd.Timestamp(first.date)).days / 365.0

# Build PricePath from underlying_price (contemporaneous spot from opprcd)
und_prices = [s.underlying_price for s in snaps]
times = [(pd.Timestamp(s.date) - pd.Timestamp(first.date)).days / 365.0 for s in snaps]
price_path_etl = PricePath(times=times, prices=und_prices)

etl_result = run_delta_hedge(
    path=price_path_etl,
    K=first.strike,
    r=0.05,
    sigma=first.implied_vol,
    n_contracts=1,
)

n_pass_etl = sum(c.invariant_holds for c in etl_result.certificates)
print("Backtest on ETL data")
print(f"  Observations     : {len(snaps)}")
print(f"  T_total (years)  : {T_total_etl:.4f}")
print(f"  Hedging cost     : ${etl_result.total_hedging_cost:>8.2f}  (1 contract)")
print(f"  Certificates     : {n_pass_etl}/{len(etl_result.certificates)} passed")
print()
print("To run on real data, replace synthetic_df above with:")
print("  df = pd.read_parquet('../data/spy_atm_options.parquet')")
print("  snaps = optionmetrics_option_snapshots_from_df(df)")
print("  (requires WRDS OptionMetrics subscription)")